# 03 — Embeddings & FAISS Vector Database
## Converting 56,617 legal chunks into searchable vectors
We will:
- Load all 56,617 chunks from JSON
- Convert text to vectors using sentence-transformers
- Build FAISS index
- Save to vector_store/
- Test with a real legal question

In [1]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import os

# Load all chunks
with open('../data/processed_chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print("✅ Chunks loaded!")
print(f"📊 Total chunks : {len(chunks)}")
print()
print("📖 Sample chunk:")
print("-" * 60)
print(f"Act Title : {chunks[0]['act_title']}")
print(f"Section   : {chunks[0]['section_id']}")
print(f"Text      : {chunks[0]['text'][:200]}")

C:\Users\mrige\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Chunks loaded!
📊 Total chunks : 56617

📖 Sample chunk:
------------------------------------------------------------
Act Title : THE BUILDING AND OTHER CONSTRUCTION WORKERS’ WELFARE CESS ACT, 1996
Section   : Section 1.
Text      : (1) This Act may be called the Building and Other Construction Workers’ Welfare Cess Act, 1996. (2) It extends to the whole of India. (3) It shall be deemed to have come into force on the 3rd day of N


## Step 1 — Load Embedding Model
Load sentence-transformer model to convert text to vectors

embedding model we use:  all-MiniLM-L6-v2

Converts text into 384 numbers

Very fast and accurate for legal text
📊 Model details:
   Model name : all-MiniLM-L6-v2
   Made by    : Microsoft --Free to use
   Vector size: 384 dimensions

In [2]:
# Load the embedding model
print("⏳ Loading embedding model...")
print("(This may take 1-2 minutes first time — downloading model)")
print()

model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded!")
print()
print("📊 Model details:")
print(f"   Model name : all-MiniLM-L6-v2")
print(f"   Made by    : Microsoft")
print(f"   Vector size: {model.get_sentence_embedding_dimension()} dimensions")
print()

# Test the model on one sentence
test_sentence = "penalty for not paying income tax"
test_vector = model.encode([test_sentence])
print(f"🧪 Test embedding:")
print(f"   Input  : '{test_sentence}'")
print(f"   Output : {test_vector.shape} shaped vector")
print(f"   First 5 numbers: {test_vector[0][:5].round(4)}")

⏳ Loading embedding model...
(This may take 1-2 minutes first time — downloading model)



C:\Users\mrige\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrige\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

✅ Embedding model loaded!

📊 Model details:
   Model name : all-MiniLM-L6-v2
   Made by    : Microsoft
   Vector size: 384 dimensions

🧪 Test embedding:
   Input  : 'penalty for not paying income tax'
   Output : (1, 384) shaped vector
   First 5 numbers: [ 0.0227  0.0977  0.011  -0.0259  0.0299]


C:\Users\mrige\AppData\Local\Temp\ipykernel_33748\3823812813.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Vector size: {model.get_sentence_embedding_dimension()} dimensions")


## Step 2 — Embed All 56,617 Chunks
Convert every legal chunk into a 384 dimensional vector
WARNING: This will take 10-20 minutes. Do not stop it!

In [3]:
# Extract just the text from all chunks
texts = [chunk['text'] for chunk in chunks]

print(f"📊 Total texts to embed : {len(texts)}")
print(f"⏳ Starting embedding...")
print(f"⚠️  This will take 10-20 minutes. Please wait!")
print()

# Embed all chunks in batches of 64
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print()
print("✅ Embedding complete!")
print(f"📊 Embeddings shape : {embeddings.shape}")
print(f"   Rows    : {embeddings.shape[0]} (one per chunk)")
print(f"   Columns : {embeddings.shape[1]} (384 dimensions each)")

📊 Total texts to embed : 56617
⏳ Starting embedding...
⚠️  This will take 10-20 minutes. Please wait!



Batches: 100%|██████████| 885/885 [09:09<00:00,  1.61it/s]



✅ Embedding complete!
📊 Embeddings shape : (56617, 384)
   Rows    : 56617 (one per chunk)
   Columns : 384 (384 dimensions each)


## Step 3 — Build FAISS Index
Store all 56,617 vectors in FAISS search engine

In [4]:
# Build FAISS index
print("⏳ Building FAISS index...")
print()

# Get vector dimension
dimension = embeddings.shape[1]
print(f"📊 Vector dimension : {dimension}")

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add all embeddings to index
index.add(embeddings.astype('float32'))

print(f"✅ FAISS index built!")
print(f"📊 Total vectors in index : {index.ntotal}")
print()
print("🔍 FAISS is now ready to search 56,617 legal chunks!")

⏳ Building FAISS index...

📊 Vector dimension : 384
✅ FAISS index built!
📊 Total vectors in index : 56617

🔍 FAISS is now ready to search 56,617 legal chunks!


## Step 4 — Save FAISS Index to Disk
Save index and metadata to vector_store/ folder

In [5]:
# Save FAISS index to disk
faiss_path = '../vector_store/legal_index.faiss'
metadata_path = '../vector_store/metadata.json'

# Save FAISS index
print("⏳ Saving FAISS index...")
faiss.write_index(index, faiss_path)
print(f"✅ FAISS index saved!")
print(f"📁 Location : {faiss_path}")
print()

# Save metadata
print("⏳ Saving metadata...")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f"✅ Metadata saved!")
print(f"📁 Location : {metadata_path}")
print()

# Check file sizes
faiss_size = os.path.getsize(faiss_path) / (1024 * 1024)
meta_size = os.path.getsize(metadata_path) / (1024 * 1024)
print(f"📊 File sizes:")
print(f"   legal_index.faiss : {faiss_size:.2f} MB")
print(f"   metadata.json     : {meta_size:.2f} MB")

⏳ Saving FAISS index...
✅ FAISS index saved!
📁 Location : ../vector_store/legal_index.faiss

⏳ Saving metadata...
✅ Metadata saved!
📁 Location : ../vector_store/metadata.json

📊 File sizes:
   legal_index.faiss : 82.94 MB
   metadata.json     : 38.04 MB


## Step 5 — Test FAISS Search
Search with a real legal question and see results

In [6]:
# Test FAISS search with a real legal question
def search_legal(query, k=5):
    # Convert query to vector
    query_vector = model.encode([query]).astype('float32')
    
    # Search FAISS
    distances, indices = index.search(query_vector, k=k)
    
    # Get results with metadata
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        results.append({
            'rank'     : i + 1,
            'distance' : round(float(distances[0][i]), 4),
            'act_title': chunk['act_title'],
            'section'  : chunk['section_id'],
            'heading'  : chunk['section_heading'],
            'text'     : chunk['text'][:200]
        })
    return results

# Test with a real question
query = "What is the penalty for income tax evasion?"
print(f"🔍 Query: '{query}'")
print("=" * 60)

results = search_legal(query, k=5)

for r in results:
    print(f"\n📌 Rank {r['rank']} (Distance: {r['distance']})")
    print(f"   Act     : {r['act_title']}")
    print(f"   Section : {r['section']}")
    print(f"   Heading : {r['heading']}")
    print(f"   Text    : {r['text']}...")
    print("-" * 60)

🔍 Query: 'What is the penalty for income tax evasion?'

📌 Rank 1 (Distance: 0.6378)
   Act     : THE BLACK MONEY (UNDISCLOSED FOREIGN INCOMEAND ASSETS) AND IMPOSITION OF TAX ACT, 2015
   Section : Section 51.
   Heading : Punishment for wilful attempt to evade tax.
   Text    : attempt to evade any tax, penalty or interest chargeable or imposable under this Act or the payment thereof shall include a case where any person—...
------------------------------------------------------------

📌 Rank 2 (Distance: 0.6805)
   Act     : THE CENTRAL GOODS AND SERVICES TAX ACT, 2017 (Last Updated on 17th June 2020)
   Section : Section 122.
   Heading : Penalty for certain offences.
   Text    : (1) Where a taxable person who–– shall be liable to pay a penalty of ten thousand rupees or an amount equivalent to the tax evaded or the tax not deducted under section 51 or short deducted or deducte...
------------------------------------------------------------

📌 Rank 3 (Distance: 0.7146)
   Act     : T

## Final Summary — FAISS Vector Database Complete

In [7]:
print("=" * 60)
print("📊 FAISS VECTOR DATABASE SUMMARY")
print("=" * 60)
print()
print("Step 1 — Data Loaded:")
print(f"   ✅ 56,617 chunks loaded from JSON")
print()
print("Step 2 — Embedding Model:")
print(f"   ✅ Model  : all-MiniLM-L6-v2 by Microsoft")
print(f"   ✅ Vectors: 384 dimensions each")
print()
print("Step 3 — FAISS Index Built:")
print(f"   ✅ Type   : IndexFlatL2")
print(f"   ✅ Vectors: 56,617 stored")
print()
print("Step 4 — Saved to Disk:")
print(f"   ✅ legal_index.faiss : 82.94 MB")
print(f"   ✅ metadata.json     : 38.04 MB")
print()
print("Step 5 — Search Tested:")
print(f"   ✅ Query  : 'penalty for income tax evasion'")
print(f"   ✅ Result : Black Money Act, GST Act, Wealth Tax Act")
print(f"   ✅ Semantic search working perfectly!")
print()
print("=" * 60)
print("✅ Ready to move to 04_rag_pipeline.ipynb!")
print("=" * 60)

📊 FAISS VECTOR DATABASE SUMMARY

Step 1 — Data Loaded:
   ✅ 56,617 chunks loaded from JSON

Step 2 — Embedding Model:
   ✅ Model  : all-MiniLM-L6-v2 by Microsoft
   ✅ Vectors: 384 dimensions each

Step 3 — FAISS Index Built:
   ✅ Type   : IndexFlatL2
   ✅ Vectors: 56,617 stored

Step 4 — Saved to Disk:
   ✅ legal_index.faiss : 82.94 MB
   ✅ metadata.json     : 38.04 MB

Step 5 — Search Tested:
   ✅ Query  : 'penalty for income tax evasion'
   ✅ Result : Black Money Act, GST Act, Wealth Tax Act
   ✅ Semantic search working perfectly!

✅ Ready to move to 04_rag_pipeline.ipynb!
